# 2.2 注意力机制优化

## 什么是注意力优化？

标准自注意力的计算复杂度为$O(n^2)$，需要存储$n \times n$的注意力矩阵，是长序列推理的核心瓶颈。注意力优化通过改进计算方式、减少KV数量或跳过不重要计算来降低延迟和内存。

## 为什么注意力是瓶颈？

标准注意力的计算过程：
$$\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中：
- $Q \in \mathbb{R}^{n \times d_k}$：查询矩阵
- $K \in \mathbb{R}^{n \times d_k}$：键矩阵
- $V \in \mathbb{R}^{n \times d_v}$：值矩阵
- $QK^T \in \mathbb{R}^{n \times n}$：注意力分数矩阵，这是$O(n^2)$的根源
- $d_k$：每个头的维度

当$n=4096$时，注意力矩阵需要$4096 \times 4096 \times 4 = 64$MB（FP32），且随序列长度平方增长。

## 注意力优化的三大方向

1. **高效计算**：Flash Attention等，不改变数学结果，优化硬件执行
2. **架构优化**：MQA/GQA等，减少KV数量，降低KV Cache和计算量
3. **稀疏计算**：跳过不重要的注意力对，降低计算复杂度

## 注意力优化在推理中的两个阶段

LLM推理分为两个阶段，注意力优化的侧重点不同：

| 阶段 | 特点 | 注意力瓶颈 | 优化重点 |
|------|------|-----------|----------|
| Prefill | 一次处理整个输入，计算密集 | $O(n^2)$注意力矩阵 | Flash Attention |
| Decode | 逐token生成，内存带宽密集 | KV Cache读取 | GQA/KV压缩 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import time
from typing import Optional, Tuple

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## 2.2.1 高效注意力计算

### Flash Attention 原理详解

#### 什么是Flash Attention？

Flash Attention通过分块计算（tiling）和重计算（recomputation）策略，减少对HBM（高带宽内存）的读写次数，将注意力计算变为IO-bound而非compute-bound。不改变数学结果，是精确注意力的硬件高效实现。

#### 理解前提：GPU的内存层次

要理解Flash Attention，必须先理解GPU的内存层次：

| 内存类型 | 大小 | 带宽 | 延迟 |
|---------|------|------|------|
| HBM（高带宽内存） | 40-80 GB | ~2 TB/s | ~100 ns |
| SRAM（片上静态内存） | ~20 MB | ~19 TB/s | ~1 ns |

关键洞察：**SRAM比HBM快约10倍，但容量小约2000倍**。标准注意力的瓶颈不是计算本身，而是需要在HBM和SRAM之间反复搬运$O(n^2)$大小的中间结果。

#### 标准注意力的内存访问问题

标准注意力的计算分为3步，每步都需要读写HBM：

```
Step 1: S = QK^T / √d_k     → 写S到HBM (N×N矩阵)
Step 2: P = softmax(S)       → 从HBM读S，写P到HBM (N×N矩阵)
Step 3: O = PV               → 从HBM读P，计算O
```

问题在于$S$和$P$都是$N \times N$矩阵，当$N=4096$时，仅$S$就需要64MB。每步都要将这个大矩阵写入HBM再读回来，内存带宽成为瓶颈——GPU的计算单元大部分时间在等数据搬运。

#### Flash Attention的核心思想：分块计算（Tiling）

Flash Attention的核心问题是：**能不能不分步写中间结果到HBM，而是在SRAM中一次性算完？**

答案是可以，但有一个关键障碍：**softmax需要看到整行数据才能计算**。

为什么？回忆softmax的定义：
$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

分母$\sum_j e^{x_j}$需要对**整行所有元素**求和。如果我们只加载了一块（block）的K和V，就只能算出一部分的$e^{x_j}$，无法得到完整的分母。

这就是**在线softmax（Online Softmax）**要解决的问题。

#### 什么是在线softmax？

在线softmax是一种**增量式**计算softmax的方法，允许我们逐块处理数据，而不需要一次性看到整行。

为了理解在线softmax，我们先看标准softmax的数值稳定性技巧，再推导在线版本。

**标准softmax的数值稳定技巧**：

直接计算$e^{x_i}$容易溢出（如$e^{1000}$），所以标准做法是减去行最大值：
$$\text{softmax}(x_i) = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}, \quad m = \max_j x_j$$

这个$m$（行最大值）的作用是**防止指数溢出**，同时不改变softmax的结果（分子分母的$e^m$约掉了）。

**在线softmax的关键洞察**：

如果我们逐块处理，每个块有自己的局部最大值$m^{(j)}$，那么当新块到来时，全局最大值可能更新为更大的值。此时之前块已经算出的$e^{x - m^{(old)}}$需要修正——因为分母的归一化基准变了。

具体来说，假设我们已经处理了第1块，得到了：
- 局部最大值 $m^{(1)}$
- 局部指数和 $\ell^{(1)} = \sum_{j \in \text{block1}} e^{x_j - m^{(1)}}$
- 局部加权和 $O^{(1)} = \sum_{j \in \text{block1}} \frac{e^{x_j - m^{(1)}}}{\ell^{(1)}} v_j$

现在处理第2块，发现全局最大值更新为$m^{(2)} > m^{(1)}$。我们需要修正第1块的结果：

**修正指数和**：
$$\ell^{(2)} = e^{m^{(1)} - m^{(2)}} \cdot \ell^{(1)} + \sum_{j \in \text{block2}} e^{x_j - m^{(2)}}$$

为什么乘以$e^{m^{(1)} - m^{(2)}}$？因为$\ell^{(1)}$是按$m^{(1)}$归一化的，现在归一化基准变成了$m^{(2)}$，需要将之前的指数和调整到新的基准：
$$\sum_{j \in \text{block1}} e^{x_j - m^{(2)}} = \sum_{j \in \text{block1}} e^{x_j - m^{(1)} + m^{(1)} - m^{(2)}} = e^{m^{(1)} - m^{(2)}} \cdot \sum_{j \in \text{block1}} e^{x_j - m^{(1)}} = e^{m^{(1)} - m^{(2)}} \cdot \ell^{(1)}$$

**修正输出**：
$$O^{(2)} = e^{m^{(1)} - m^{(2)}} \cdot O^{(1)} + \sum_{j \in \text{block2}} e^{x_j - m^{(2)}} v_j$$

注意这里的$O$存储的是**未归一化的加权和**（分子部分），最终再除以$\ell$得到归一化结果。

#### 为什么需要最大值？

行最大值$m$有两个作用：

1. **数值稳定性**：$e^{x}$当$x$很大时会溢出。减去最大值后，$x - m \leq 0$，所以$e^{x-m} \leq 1$，永远不会溢出。

2. **在线修正的桥梁**：当新块到来导致最大值从$m^{(old)}$更新为$m^{(new)}$时，$e^{m^{(old)} - m^{(new)}}$就是修正因子——它将之前按旧最大值归一化的结果，调整到新的最大值基准下。**没有最大值，就无法将不同块的结果统一到同一个归一化基准下。**

#### Flash Attention前向传播的完整计算流程

将K和V按列分成$T$块（每块$B_c$列），Q按行分成$T_r$块（每块$B_r$行）：

```
初始化:
  O = 0          # 输出累加器 (未归一化)
  ℓ = 0          # 指数和累加器
  m = -∞         # 行最大值

对外层循环 (遍历K/V块 j = 1, 2, ..., T):
  1. 加载 K^(j), V^(j) 到SRAM
  2. 计算 S^(j) = Q @ K^(j)^T / √d_k    ← 只算Q与当前K块的分数
  3. 更新行最大值: m^(new) = max(m^(old), max(S^(j)))
  4. 计算修正因子: correction = exp(m^(old) - m^(new))
  5. 修正之前的累加结果:
     ℓ = ℓ * correction + sum(exp(S^(j) - m^(new)))
     O = O * correction + exp(S^(j) - m^(new)) @ V^(j)
  6. 更新 m = m^(new)

最终归一化:
  O = O / ℓ
```

**关键点**：
- 每次只加载一小块K和V到SRAM，计算完就丢弃，**不需要存储完整的$N \times N$注意力矩阵**
- 通过维护$m$、$\ell$、$O$三个运行统计量，逐步累积结果，最终归一化
- 修正因子$e^{m^{(old)} - m^{(new)}}$确保了之前块的结果在新最大值下仍然正确

#### 用一个具体例子理解在线softmax

假设一行分数为 $[2, 1, 4, 3]$，分成两块：块1=$[2, 1]$，块2=$[4, 3]$。

**标准softmax**（需要看到全部数据）：
$$m = 4, \quad \text{softmax} = \frac{[e^{-2}, e^{-3}, e^0, e^{-1}]}{e^{-2}+e^{-3}+e^0+e^{-1}}$$

**在线softmax**（逐块处理）：

处理块1：
- $m^{(1)} = \max(2, 1) = 2$
- $\ell^{(1)} = e^{2-2} + e^{1-2} = 1 + 0.368 = 1.368$
- $O^{(1)}$ = 未归一化加权和（暂存）

处理块2：
- $m^{(2)} = \max(m^{(1)}, \max(4, 3)) = \max(2, 4) = 4$
- 修正因子 = $e^{m^{(1)} - m^{(2)}} = e^{2-4} = e^{-2} = 0.135$
- $\ell^{(2)} = 0.135 \times 1.368 + (e^{4-4} + e^{3-4}) = 0.185 + (1 + 0.368) = 1.553$
- 验证：$e^{-2}+e^{-3}+e^0+e^{-1} = 0.135+0.050+1+0.368 = 1.553$ ✓

最终归一化：$O = O^{(2)} / \ell^{(2)}$，与标准softmax结果完全一致。

**核心要点**：修正因子$e^{m^{(old)} - m^{(new)}}$将之前按旧最大值2归一化的指数和，调整到新最大值4的基准下，确保了数学等价性。

In [ ]:
print(f"=== 在线softmax逐步演示 ===")
scores = torch.tensor([2.0, 1.0, 4.0, 3.0])
block_size_demo = 2

standard_result = F.softmax(scores, dim=0)
print(f"输入分数: {scores.tolist()}")
print(f"标准softmax结果: {[f'{x:.4f}' for x in standard_result.tolist()]}")
print(f"\n--- 在线softmax逐步过程 ---")

m = float('-inf')
l = 0.0

for j in range(0, len(scores), block_size_demo):
    block = scores[j:j+block_size_demo]
    print(f"\n处理块{j//block_size_demo + 1}: scores[{j}:{j+block_size_demo}] = {block.tolist()}")

    m_old = m
    m_new = max(m_old, block.max().item())
    print(f"  旧最大值 m^(old) = {m_old}, 本块最大值 = {block.max().item():.1f}")
    print(f"  新最大值 m^(new) = {m_new}")

    correction = math.exp(m_old - m_new) if m_old != float('-inf') else 0.0
    print(f"  修正因子 e^(m_old - m_new) = e^({m_old} - {m_new}) = {correction:.4f}")

    block_exp = torch.exp(block - m_new)
    print(f"  本块指数 e^(scores - m_new) = {[f'{x:.4f}' for x in block_exp.tolist()]}")

    l_new = correction * l + block_exp.sum().item()
    print(f"  ℓ^(new) = {correction:.4f} × {l:.4f} + {block_exp.sum().item():.4f} = {l_new:.4f}")

    m = m_new
    l = l_new

online_result = torch.exp(scores - m) / l
print(f"\n最终归一化: softmax = e^(scores - {m}) / {l:.4f}")
print(f"在线softmax结果: {[f'{x:.4f}' for x in online_result.tolist()]}")
print(f"标准softmax结果: {[f'{x:.4f}' for x in standard_result.tolist()]}")
print(f"最大差异: {(online_result - standard_result).abs().max().item():.10f}")
print(f"\n结论: 在线softmax与标准softmax结果完全一致，但可以逐块处理，无需存储完整注意力矩阵")

In [ ]:
def standard_attention(Q, K, V):
    """标准注意力实现 - O(N²)内存"""
    head_dim = Q.shape[-1]
    attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (head_dim ** 0.5)
    attn_probs = F.softmax(attn_weights, dim=-1)
    output = torch.matmul(attn_probs, V)
    return output, attn_weights

def flash_attention_tiled(Q, K, V, block_size=64):
    """Flash Attention分块计算原理演示
    分块计算softmax，避免存储完整N×N注意力矩阵

    逐步对应上面的数学公式：
    - row_max 对应 m^(j)：行最大值
    - row_sum 对应 ℓ^(j)：指数和（归一化因子）
    - output  对应 O^(j)：未归一化的加权和
    - exp_correction 对应 e^(m^(old) - m^(new))：修正因子
    """
    B, H, N, D = Q.shape
    output = torch.zeros_like(Q)                              # O^(0) = 0
    row_max = torch.full((B, H, N, 1), float('-inf'), device=Q.device)  # m^(0) = -∞
    row_sum = torch.zeros((B, H, N, 1), device=Q.device)     # ℓ^(0) = 0

    for j in range(0, N, block_size):
        K_block = K[:, :, j:j+block_size, :]                 # 加载第j个K块到SRAM
        V_block = V[:, :, j:j+block_size, :]                 # 加载第j个V块到SRAM
        S_block = torch.matmul(Q, K_block.transpose(-2, -1)) / (D ** 0.5)  # S^(j) = Q @ K^(j)^T / √d

        block_max = S_block.max(dim=-1, keepdim=True).values  # 当前块的最大值
        new_max = torch.maximum(row_max, block_max)           # m^(j) = max(m^(j-1), max(S^(j)))
        exp_correction = torch.exp(row_max - new_max)         # e^(m^(j-1) - m^(j))：修正因子
        block_exp = torch.exp(S_block - new_max)              # e^(S^(j) - m^(j))：当前块的指数值

        row_sum = row_sum * exp_correction + block_exp.sum(dim=-1, keepdim=True)  # ℓ^(j) 更新
        output = output * exp_correction + torch.matmul(block_exp, V_block)       # O^(j) 更新
        row_max = new_max                                      # 更新行最大值

    output = output / row_sum                                  # 最终归一化: O = O / ℓ
    return output

B, H, N, D = 1, 4, 256, 64
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

out_standard, attn_weights = standard_attention(Q, K, V)
out_flash = flash_attention_tiled(Q, K, V, block_size=64)

diff = (out_standard - out_flash).norm() / out_standard.norm() * 100

standard_mem = B * H * N * N * 4
flash_mem = B * H * N * D * 4 * 3

print(f"=== Flash Attention vs 标准注意力 ===")
print(f"序列长度: {N}, 头数: {H}, 头维度: {D}")
print(f"输出差异: {diff:.6f}% (数学等价)")
print(f"\n内存占用:")
print(f"  标准注意力(N×N矩阵): {standard_mem/1024:.1f} KB")
print(f"  Flash Attention(无N×N): {flash_mem/1024:.1f} KB")
print(f"  节省: {(1-flash_mem/standard_mem)*100:.1f}%")

print(f"\n=== 不同序列长度的内存对比 ===")
for seq_len in [256, 512, 1024, 2048, 4096, 8192]:
    std_mem = B * H * seq_len * seq_len * 4 / 1024 / 1024
    fl_mem = B * H * seq_len * D * 4 * 3 / 1024 / 1024
    print(f"  N={seq_len:<6} 标准={std_mem:.1f}MB Flash={fl_mem:.1f}MB 节省{(1-fl_mem/std_mem)*100:.0f}%")

### Flash Attention 反向传播详解

#### 为什么反向传播也需要优化？

训练LLM时，反向传播需要计算$\frac{\partial L}{\partial Q}$、$\frac{\partial L}{\partial K}$、$\frac{\partial L}{\partial V}$。标准实现中，反向传播需要读取前向传播存储的注意力概率矩阵$P = \text{softmax}(S)$，这是一个$N \times N$矩阵，需要大量HBM读写。

#### 标准注意力反向传播的推导

设$O = PV$，$P = \text{softmax}(S)$，$S = QK^T / \sqrt{d_k}$。已知上游梯度$\frac{\partial L}{\partial O}$（简记为$dO$），需要求$dQ$、$dK$、$dV$。

**第一步：求$dV$**

$O = PV$对$V$求偏导：
$$dV = P^T \cdot dO$$

直觉理解：每个输出$O_i$是$P_i$（第$i$行的注意力权重）与$V$的加权和，所以$V$的梯度是所有行的注意力权重对$dO$的加权贡献。

**第二步：求$dP$**

$O = PV$对$P$求偏导：
$$dP = dO \cdot V^T$$

**第三步：从$dP$求$dS$（最关键的一步）**

$P = \text{softmax}(S)$，softmax的雅可比矩阵为：
$$\frac{\partial P_{ij}}{\partial S_{ik}} = P_{ij}(\delta_{jk} - P_{ik})$$

利用这个性质，可以推导出：
$$dS = P \odot (dP - D), \quad D = \text{rowsum}(dO \odot O)$$

其中$D_i = \sum_j dP_{ij} P_{ij}$是softmax归一化的修正项。关键等式：
$$D_i = \sum_j (dO_i V_j^T) P_{ij} = dO_i (\sum_j P_{ij} V_j)^T = dO_i O_i^T = \text{rowsum}(dO_i \odot O_i)$$

这个等式意味着我们**不需要先计算完整的$dP$再求和**，而是直接用$dO$和$O$计算$D$，然后分块计算$dS$。这是Flash Attention反向传播能分块进行的关键。

- $dP \odot P$：直接将梯度乘以注意力权重
- 减去$D$：修正softmax的归一化效应（因为softmax的分母也依赖于$S$）
- 最终$dS$的含义：分数$S$的微小变化如何影响损失

**第四步：从$dS$求$dQ$、$dK$**

$S = QK^T / \sqrt{d_k}$，所以：
$$dQ = dS \cdot K / \sqrt{d_k}$$
$$dK = dS^T \cdot Q / \sqrt{d_k}$$

#### 标准反向传播的内存瓶颈

标准实现需要：
1. **存储$P$矩阵**：前向传播时将$P$写入HBM，反向传播时读回，占用$N \times N$内存
2. **读取$P$矩阵**：计算$dS$时需要$P$，需要从HBM读取

当$N=4096$时，$P$矩阵需要64MB（FP32），多个头和层叠加后内存需求巨大。

#### Flash Attention反向传播：重计算（Recomputation）

Flash Attention反向传播的核心思想：**不存储$P$矩阵，而是在需要时从$Q$、$K$、$V$重新计算$S$和$P$**。

```
标准反向传播:
  前向: 存储 P 到 HBM (N×N)
  反向: 从 HBM 读取 P → 计算 dS, dV, dQ, dK
  内存: O(N²) 用于存储 P

Flash Attention反向传播:
  前向: 只存储 O, m, ℓ 到 HBM (O(N))
  反向: 分块重算 S^(j) 和 P^(j) → 计算 dS^(j), dV^(j), dQ^(j), dK^(j)
  内存: O(N) 用于存储 O, m, ℓ
```

具体流程：

```
对每个K/V块 j = 1, 2, ..., T:
  1. 加载 Q, K^(j), V^(j) 到SRAM
  2. 重算 S^(j) = Q @ K^(j)^T / √d_k
  3. 重算 P^(j) = softmax(S^(j), m, ℓ)  ← 使用前向存储的 m, ℓ
  4. 计算 D = rowsum(dO ⊙ O)            ← 关键: 用dO和O计算修正项
  5. 计算 dP^(j) = dO @ V^(j)^T
  6. 计算 dS^(j) = P^(j) ⊙ (dP^(j) - D) ← D已预计算，可分块使用
  7. 计算 dV^(j) = P^(j)^T @ dO
  8. 计算 dQ += dS^(j) @ K^(j) / √d_k
  9. 计算 dK^(j) = dS^(j)^T @ Q / √d_k
```

#### 为什么重计算反而更快？

这看似矛盾——多算了$S$和$P$，怎么反而更快？关键在于**计算速度远快于内存读写**：

| 操作 | 耗时 |
|------|------|
| 从HBM读取$P$（$N^2$个元素） | $O(N^2 / \text{带宽})$ |
| 重算$S$和$P$（$N^2 d$次FLOPs） | $O(N^2 d / \text{算力})$ |

当$N$较大时，HBM带宽成为瓶颈，而GPU的计算单元大量空闲。重计算利用了这些空闲的计算能力，**用计算换带宽**：
- 节省的HBM读写时间 > 重计算的额外计算时间
- 还省下了存储$P$的$O(N^2)$内存
- Flash Attention-2论文报告：反向传播重计算仅增加约5-10%的FLOPs，但节省了2-4倍的HBM访问

#### 前向传播存储了什么？

Flash Attention前向传播只存储以下内容到HBM（用于反向传播）：

| 存储项 | 大小 | 用途 |
|--------|------|------|
| $O$ | $N \times d$ | 输出，反向传播需要$dO$ |
| $m$ | $N \times 1$ | 行最大值，重算$P$时用于数值稳定 |
| $\ell$ | $N \times 1$ | 指数和，重算$P$时用于归一化 |
| $Q, K, V$ | $3 \times N \times d$ | 输入，重算$S$和$P$需要 |

总计$O(Nd)$，而非标准实现的$O(N^2)$。当$N=4096, d=64$时，存储量从$16M$元素降到$~0.3M$元素，减少约50倍。

In [ ]:
def standard_attention_backward(dO, Q, K, V, P):
    """标准注意力反向传播：需要存储的P矩阵

    Args:
        dO: 上游梯度 (B, H, N, D)
        Q, K, V: 前向传播的输入
        P: 前向传播存储的注意力概率矩阵 (B, H, N, N)
    """
    d_k = Q.shape[-1]
    dV = torch.matmul(P.transpose(-2, -1), dO)
    dP = torch.matmul(dO, V.transpose(-2, -1))
    dS = P * (dP - (dP * P).sum(dim=-1, keepdim=True))
    dQ = torch.matmul(dS, K) / (d_k ** 0.5)
    dK = torch.matmul(dS.transpose(-2, -1), Q) / (d_k ** 0.5)
    return dQ, dK, dV

def flash_attention_backward(dO, Q, K, V, O, row_max, row_sum, block_size=64):
    """Flash Attention反向传播：不存储P，分块重算

    核心数学推导：
    dS_ij = P_ij * (dP_ij - D_i)
    其中 D_i = sum_j(dP_ij * P_ij) = (dO_i * O_i).sum()
    这个等式来自：sum_j(dO_i @ V_j^T * P_ij) = dO_i @ (sum_j P_ij * V_j)^T = dO_i @ O_i^T
    这样无需先算完整dP再求和，直接用dO和O计算D_i即可。

    Args:
        dO: 上游梯度 (B, H, N, D)
        Q, K, V: 前向传播的输入
        O: 前向传播的输出 (B, H, N, D)
        row_max: 前向传播存储的行最大值 (B, H, N, 1)
        row_sum: 前向传播存储的指数和 (B, H, N, 1)
    """
    B, H, N, D = Q.shape
    dQ = torch.zeros_like(Q)
    dK = torch.zeros_like(K)
    dV = torch.zeros_like(V)
    Di = (dO * O).sum(dim=-1, keepdim=True)

    for j in range(0, N, block_size):
        K_block = K[:, :, j:j+block_size, :]
        V_block = V[:, :, j:j+block_size, :]
        S_block = torch.matmul(Q, K_block.transpose(-2, -1)) / (D ** 0.5)
        P_block = torch.exp(S_block - row_max) / row_sum

        dV_block = torch.matmul(P_block.transpose(-2, -1), dO)
        dP_block = torch.matmul(dO, V_block.transpose(-2, -1))
        dS_block = P_block * (dP_block - Di)

        dQ += torch.matmul(dS_block, K_block) / (D ** 0.5)
        dK_block = torch.matmul(dS_block.transpose(-2, -1), Q) / (D ** 0.5)

        dK[:, :, j:j+block_size, :] = dK_block
        dV[:, :, j:j+block_size, :] = dV_block

    return dQ, dK, dV

B, H, N, D = 1, 4, 64, 32
Q = torch.randn(B, H, N, D, requires_grad=True)
K = torch.randn(B, H, N, D, requires_grad=True)
V = torch.randn(B, H, N, D, requires_grad=True)

O_standard, P = standard_attention(Q, K, V)
loss = O_standard.sum()
loss.backward()
dQ_ref, dK_ref, dV_ref = Q.grad.clone(), K.grad.clone(), V.grad.clone()

Q.grad, K.grad, V.grad = None, None, None
with torch.no_grad():
    O_flash = flash_attention_tiled(Q, K, V, block_size=32)
    row_max = torch.full((B, H, N, 1), float('-inf'))
    row_sum = torch.zeros((B, H, N, 1))
    for j in range(0, N, 32):
        K_b = K[:, :, j:j+32, :]
        S_b = torch.matmul(Q, K_b.transpose(-2, -1)) / (D ** 0.5)
        b_max = S_b.max(dim=-1, keepdim=True).values
        new_max = torch.maximum(row_max, b_max)
        row_sum = row_sum * torch.exp(row_max - new_max) + torch.exp(S_b - new_max).sum(dim=-1, keepdim=True)
        row_max = new_max

    dO = torch.ones_like(O_flash)
    dQ_flash, dK_flash, dV_flash = flash_attention_backward(
        dO, Q, K, V, O_flash, row_max, row_sum, block_size=32
    )

print(f"=== Flash Attention 反向传播验证 ===")
print(f"序列长度: {N}, 头数: {H}, 头维度: {D}")
print(f"\n梯度对比 (Flash反向 vs PyTorch自动微分):")
for name, ref, flash in [('dQ', dQ_ref, dQ_flash), ('dK', dK_ref, dK_flash), ('dV', dV_ref, dV_flash)]:
    rel_diff = (ref - flash).norm() / ref.norm() * 100
    print(f"  {name}: 相对差异 = {rel_diff:.4f}% ({'PASS' if rel_diff < 1 else 'FAIL'})")

standard_storage = B * H * N * N
flash_storage = B * H * N * 3
print(f"\n=== 内存对比 (前向传播需要存储的元素数) ===")
print(f"标准实现 (存储P矩阵): {standard_storage:,} 元素")
print(f"Flash实现 (存储O+m+ℓ): {flash_storage:,} 元素")
print(f"节省: {(1 - flash_storage / standard_storage) * 100:.1f}%")

print(f"\n=== 不同序列长度的反向传播内存对比 ===")
print(f"{'N':<8} {'标准(P矩阵)':<18} {'Flash(O+m+ℓ)':<18} {'节省':<10}")
print("-" * 54)
for seq_len in [256, 512, 1024, 2048, 4096]:
    std = B * H * seq_len * seq_len
    fl = B * H * seq_len * 3
    print(f"{seq_len:<8} {std/1e6:<18.2f}M {fl/1e3:<18.1f}K {(1-fl/std)*100:<10.1f}%")

### 产业级实战：PyTorch SDPA（scaled_dot_product_attention）

PyTorch 2.0+ 提供了 `F.scaled_dot_product_attention` (SDPA)，它自动选择最优后端：
- **Flash Attention** (CUDA, 计算密集型)
- **Memory-Efficient Attention** (xFormers风格, 内存密集型)
- **Math Implementation** (回退实现)

这是产业界最常用的注意力优化方式——无需安装额外库，一行代码即可获得Flash Attention级别的性能。

#### SDPA的自动后端选择逻辑

SDPA根据输入的形状和数据类型自动选择最优后端：

1. **Flash Attention后端**：当head_dim ≤ 128、数据类型为FP16/BF16、CUDA可用时优先选择。Flash Attention-2要求head_dim是8的倍数。

2. **Memory-Efficient后端**：当Flash Attention不可用时（如head_dim过大、不支持的数据类型），使用xFormers风格的内存高效实现。同样不存储$N \times N$注意力矩阵，但实现策略略有不同。

3. **Math后端**：当以上两个都不可用时（如CPU上运行），回退到标准数学实现，计算完整注意力矩阵。

可以通过 `torch.backends.cuda.sdp_kernel` 上下文管理器手动控制后端选择。

In [ ]:
def manual_attention(q, k, v, attn_mask=None):
    """手动实现的标准注意力（用于对比基准）"""
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if attn_mask is not None:
        scores = scores.masked_fill(attn_mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, v)

def sdpa_attention(q, k, v, attn_mask=None):
    """PyTorch SDPA实现（自动选择最优后端）"""
    if attn_mask is not None:
        attn_mask = attn_mask.to(dtype=q.dtype)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)

configs = [
    (4, 512, 64),
    (4, 1024, 64),
    (4, 2048, 64),
]

print(f"=== PyTorch SDPA vs 手动注意力 性能对比 ===")
print(f"{'Batch×Heads':<14} {'SeqLen':<8} {'手动(ms)':<12} {'SDPA(ms)':<12} {'加速比':<10}")
print("-" * 56)

for n_heads, seq_len, head_dim in configs:
    q = torch.randn(1, n_heads, seq_len, head_dim)
    k = torch.randn(1, n_heads, seq_len, head_dim)
    v = torch.randn(1, n_heads, seq_len, head_dim)

    with torch.no_grad():
        for _ in range(5):
            manual_attention(q, k, v)
        start = time.perf_counter()
        for _ in range(20):
            manual_attention(q, k, v)
        manual_time = (time.perf_counter() - start) / 20 * 1000

        for _ in range(5):
            sdpa_attention(q, k, v)
        start = time.perf_counter()
        for _ in range(20):
            sdpa_attention(q, k, v)
        sdpa_time = (time.perf_counter() - start) / 20 * 1000

    speedup = manual_time / sdpa_time
    print(f"{n_heads:<14} {seq_len:<8} {manual_time:<12.2f} {sdpa_time:<12.2f} {speedup:<10.2f}x")

with torch.no_grad():
    q = torch.randn(1, 4, 64, 64)
    k = torch.randn(1, 4, 64, 64)
    v = torch.randn(1, 4, 64, 64)
    out_manual = manual_attention(q, k, v)
    out_sdpa = sdpa_attention(q, k, v)
    max_diff = (out_manual - out_sdpa).abs().max().item()

print(f"\n数值一致性: 最大差异 = {max_diff:.8f} ({'PASS' if max_diff < 1e-5 else 'FAIL'})")

print(f"\n=== SDPA后端控制 ===")
if torch.cuda.is_available():
    with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_mem_efficient=True, enable_math=True):
        print("GPU模式: Flash Attention + Mem Efficient + Math 均可用")
else:
    print("CPU模式: Math实现 (与手动等价)")

print(f"\n产业界注意力优化最佳实践:")
print(f"1. 首选: F.scaled_dot_product_attention (自动选最优后端)")
print(f"2. GPU: 自动使用Flash Attention (2-4x加速)")
print(f"3. CPU: 使用Math实现 (与手动等价)")
print(f"4. 长序列: 配合GQA减少KV → 再用SDPA加速")
print(f"5. 自定义: torch.backends.cuda.sdp_kernel 控制后端选择")

---
## 2.2.2 注意力架构优化

### MQA / GQA 原理与实现

#### 什么是MQA和GQA？

- **MHA (Multi-Head Attention)**：每个头有独立的Q/K/V投影，$h$个头有$h$组KV
- **MQA (Multi-Query Attention)**：所有头共享K/V投影，仅Q保持多头，KV只有1组
- **GQA (Grouped-Query Attention)**：Q头分组，每组共享K/V，在MHA和MQA间取折中

#### 为什么需要MQA/GQA？Decode阶段的KV Cache瓶颈

理解MQA/GQA的动机，必须从**Decode阶段的内存带宽瓶颈**出发。

Decode阶段每生成一个token，都需要：
1. 将新token通过K/V投影，得到1组新的K和V向量
2. 将新K/V拼接到KV Cache中
3. **读取完整KV Cache**（包含所有历史token的K和V）
4. 计算注意力：$\text{softmax}(q \cdot K^T / \sqrt{d_k}) \cdot V$

第3步是瓶颈——KV Cache越大，读取时间越长。KV Cache的大小与KV头数成正比：
$$\text{KV Cache大小} = 2 \times n_{kv\_heads} \times L \times d_k \times \text{dtype\_size}$$

其中$L$是序列长度。当$n_{kv\_heads} = n_{heads}$（MHA）时，KV Cache很大；减少$n_{kv\_heads}$就能线性减少KV Cache。

#### MQA/GQA的数学公式

**MHA**：$h$个头各自计算
$$\text{head}_i = \text{Attn}(QW_i^Q, KW_i^K, VW_i^V), \quad i = 1, ..., h$$

**MQA**：所有头共享KV
$$\text{head}_i = \text{Attn}(QW_i^Q, KW^K, VW^V), \quad i = 1, ..., h$$

**GQA**：$g$组共享KV，每组$h/g$个Q头
$$\text{head}_i = \text{Attn}(QW_i^Q, KW_{\lfloor i \cdot g / h \rfloor}^K, VW_{\lfloor i \cdot g / h \rfloor}^V)$$

其中：
- $h$：Q头数（总头数）
- $g$：KV头组数（GQA），MQA时$g=1$，MHA时$g=h$
- $W_i^Q \in \mathbb{R}^{d \times d_k}$：第$i$个Q头的投影矩阵
- $W_j^K, W_j^V$：第$j$组KV的投影矩阵
- KV Cache大小：$2 \times g \times L \times d_k$（与Q头数无关）

#### GQA的分组映射：Q头如何共享KV头？

假设$h=8$个Q头，$g=2$个KV头，则每个KV头服务$8/2=4$个Q头：

```
Q头:  [0, 1, 2, 3]  [4, 5, 6, 7]
        ↕              ↕
KV头:     KV_0           KV_1
```

Q头0-3都使用KV_0的K和V，Q头4-7都使用KV_1的K和V。在代码中，这通过`repeat_interleave`或`expand`实现——将KV头复制到对应的Q头维度。

#### MQA/GQA为什么不影响精度？

直觉上，共享KV似乎会损失信息，但实际影响很小，原因如下：

1. **Q仍然保持多头**：每个Q头有不同的投影矩阵，可以从共享的KV中提取不同的信息。注意力权重的多样性主要来自Q的多样性，而非KV的多样性。

2. **KV的信息冗余**：相邻Q头学到的K/V投影往往相似，共享后相当于一种正则化，减少冗余参数。

3. **GQA的折中设计**：GQA论文（Ainslie et al. 2023）的实验表明，8组KV头的GQA在大多数任务上与MHA精度相当，而1组KV头的MQA在某些任务上有轻微下降。

#### KV投影矩阵的参数量对比

| 方法 | K投影参数 | V投影参数 | 总KV参数 | 相比MHA |
|------|----------|----------|---------|--------|
| MHA | $h \times d \times d_k$ | $h \times d \times d_k$ | $2hd d_k$ | 基准 |
| MQA | $d \times d_k$ | $d \times d_k$ | $2d d_k$ | 减少$h$倍 |
| GQA | $g \times d \times d_k$ | $g \times d \times d_k$ | $2gd d_k$ | 减少$h/g$倍 |

In [ ]:
class MultiHeadAttention(nn.Module):
    """标准多头注意力 (MHA)"""
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out), k, v

class MultiQueryAttention(nn.Module):
    """多查询注意力 (MQA): 所有头共享K/V"""
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, self.head_dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, 1, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, 1, self.head_dim).transpose(1, 2)
        k = k.expand(-1, self.n_heads, -1, -1)
        v = v.expand(-1, self.n_heads, -1, -1)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out), k, v

class GroupedQueryAttention(nn.Module):
    """分组查询注意力 (GQA): Q头分组，每组共享K/V"""
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out), k, v

dim, n_heads = 512, 8
seq_len = 128
x = torch.randn(1, seq_len, dim)

mha = MultiHeadAttention(dim, n_heads)
mqa = MultiQueryAttention(dim, n_heads)
gqa = GroupedQueryAttention(dim, n_heads, n_kv_heads=2)

with torch.no_grad():
    out_mha, k_mha, v_mha = mha(x)
    out_mqa, k_mqa, v_mqa = mqa(x)
    out_gqa, k_gqa, v_gqa = gqa(x)

mha_kv_params = dim * n_heads * 2
mqa_kv_params = dim * (dim // n_heads) * 2
gqa_kv_params = dim * (2 * (dim // n_heads)) * 2

mha_kv_cache = 1 * n_heads * seq_len * (dim // n_heads) * 2 * 2
mqa_kv_cache = 1 * 1 * seq_len * (dim // n_heads) * 2 * 2
gqa_kv_cache = 1 * 2 * seq_len * (dim // n_heads) * 2 * 2

print(f"=== MHA vs MQA vs GQA 对比 ===")
print(f"dim={dim}, n_heads={n_heads}, GQA n_kv_heads=2")
print(f"\n{'方法':<8} {'KV参数量':<15} {'KV Cache/seq_len':<20} {'KV Cache节省':<15}")
print("-" * 58)
print(f"{'MHA':<8} {mha_kv_params:<15,} {mha_kv_cache/1024:.1f} KB/tok   {'基准':<15}")
print(f"{'MQA':<8} {mqa_kv_params:<15,} {mqa_kv_cache/1024:.1f} KB/tok   {(1-mqa_kv_cache/mha_kv_cache)*100:.0f}%")
print(f"{'GQA':<8} {gqa_kv_params:<15,} {gqa_kv_cache/1024:.1f} KB/tok   {(1-gqa_kv_cache/mha_kv_cache)*100:.0f}%")

print(f"\n=== Decode阶段KV Cache读取带宽分析 ===")
print(f"假设: seq_len=4096, head_dim=128, FP16, 32层")
n_layers = 32
for name, n_kv in [('MHA', 32), ('GQA(8组)', 8), ('MQA', 1)]:
    kv_bytes = n_layers * 2 * n_kv * 4096 * 128 * 2
    bandwidth_gb = kv_bytes / 1e9
    print(f"  {name:<12} KV Cache={bandwidth_gb:.2f}GB 每步需读取{bandwidth_gb:.2f}GB")
print(f"\nMQA/GQA的核心价值: 减少Decode阶段每步的KV Cache读取量")

### 产业级GQA实现：与SDPA集成

产业界（如LLaMA-2/3、Mistral）的GQA实现不使用`repeat_interleave`扩展KV头（浪费内存），而是直接在KV头维度上计算注意力，通过`expand`让Q头共享KV。以下实现更接近真实框架代码。

#### expand vs repeat_interleave的区别

- `repeat_interleave`：**实际复制数据**，内存占用增加$n_{rep}$倍
- `expand`：**不复制数据**，只创建视图（stride为0），内存占用不变

在推理中，KV Cache是最大的内存消耗，使用`expand`避免复制KV头可以节省大量内存。

In [ ]:
class EfficientGQA(nn.Module):
    """产业级GQA实现：不扩展KV，直接在KV头维度计算注意力

    与上面GroupedQueryAttention的数学等价，但内存效率更高：
    - 不调用repeat_interleave，避免KV头复制
    - 使用SDPA加速注意力计算
    - KV Cache只存储n_kv_heads份，推理时节省内存
    """
    def __init__(self, dim: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads

        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * self.head_dim, dim, bias=False)

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """将KV头扩展以匹配Q头数，使用expand而非repeat（不复制内存）"""
        if self.n_rep == 1:
            return x
        B, n_kv_heads, seq_len, head_dim = x.shape
        x = x[:, :, None, :, :].expand(B, n_kv_heads, self.n_rep, seq_len, head_dim)
        return x.reshape(B, n_kv_heads * self.n_rep, seq_len, head_dim)

    def forward(
        self,
        x: torch.Tensor,
        cached_k: Optional[torch.Tensor] = None,
        cached_v: Optional[torch.Tensor] = None,
        use_sdpa: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k_new = self.k_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v_new = self.v_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if cached_k is not None and cached_v is not None:
            k_full = torch.cat([cached_k, k_new], dim=2)
            v_full = torch.cat([cached_v, v_new], dim=2)
        else:
            k_full = k_new
            v_full = v_new

        k_expanded = self._repeat_kv(k_full)
        v_expanded = self._repeat_kv(v_full)

        if use_sdpa:
            is_causal = (cached_k is None) and (N > 1)
            attn_output = F.scaled_dot_product_attention(
                q, k_expanded, v_expanded, is_causal=is_causal
            )
        else:
            attn = (q @ k_expanded.transpose(-2, -1)) * (self.head_dim ** -0.5)
            if cached_k is None and N > 1:
                causal_mask = torch.triu(
                    torch.ones(N, N, device=x.device, dtype=torch.bool), diagonal=1
                )
                attn = attn.masked_fill(causal_mask, float('-inf'))
            attn = attn.softmax(dim=-1)
            attn_output = attn @ v_expanded

        out = attn_output.transpose(1, 2).reshape(B, N, -1)
        return self.out_proj(out), k_full, v_full

dim, n_heads, n_kv_heads = 512, 8, 2
gqa_eff = EfficientGQA(dim, n_heads, n_kv_heads)

print(f"=== 产业级GQA实现 ===")
print(f"dim={dim}, n_heads={n_heads}, n_kv_heads={n_kv_heads}")
print(f"n_rep (每个KV头服务多少Q头): {n_heads // n_kv_heads}")

x_prefill = torch.randn(1, 16, dim)
with torch.no_grad():
    out, k_cached, v_cached = gqa_eff(x_prefill, use_sdpa=True)
print(f"\nPrefill: 输入shape={x_prefill.shape}, 输出shape={out.shape}")
print(f"KV Cache: K={k_cached.shape}, V={v_cached.shape}")
print(f"KV Cache仅存储{n_kv_heads}个头 (而非{n_heads}个), 节省{(1-n_kv_heads/n_heads)*100:.0f}%")

x_decode = torch.randn(1, 1, dim)
with torch.no_grad():
    out2, k_updated, v_updated = gqa_eff(x_decode, cached_k=k_cached, cached_v=v_cached, use_sdpa=True)
print(f"\nDecode: 输入shape={x_decode.shape}, 输出shape={out2.shape}")
print(f"KV Cache更新: K={k_updated.shape}, V={v_updated.shape}")

print(f"\n=== 真实模型GQA配置对比 ===")
models = [
    ("LLaMA-2-70B", 64, 8, 128),
    ("LLaMA-3-8B", 32, 8, 128),
    ("LLaMA-3-70B", 64, 8, 128),
    ("Mistral-7B", 32, 8, 128),
    ("Mixtral-8x7B", 32, 8, 128),
]
print(f"{'模型':<20} {'Q头数':<8} {'KV头数':<8} {'头维度':<8} {'KV Cache节省':<12}")
print("-" * 56)
for name, q_heads, kv_heads, hd in models:
    saving = (1 - kv_heads / q_heads) * 100
    print(f"{name:<20} {q_heads:<8} {kv_heads:<8} {hd:<8} {saving:.0f}%")

---
## 2.2.3 滑动窗口注意力

### 什么是滑动窗口注意力？

每个token只关注最近$W$个token的KV，超出窗口的KV被丢弃。注意力计算复杂度从$O(n^2)$降为$O(n \cdot W)$，KV Cache大小固定为$O(W)$，不随序列增长。Mistral、Gemini等模型采用此策略。

### 为什么滑动窗口有效？

#### 1. 内存可控：KV Cache大小固定

标准注意力中，KV Cache随序列长度线性增长：
$$\text{KV Cache} = 2 \times n_{kv\_heads} \times L \times d_k \times \text{dtype\_size}$$

当$L=8192$时，32层、8个KV头、128维、FP16的模型需要：
$$32 \times 2 \times 8 \times 8192 \times 128 \times 2 = 1\text{GB}$$

滑动窗口将$L$替换为$W$，KV Cache固定为：
$$32 \times 2 \times 8 \times 4096 \times 128 \times 2 = 512\text{MB}$$

无论输入多长，KV Cache不会超过这个上限。

#### 2. 局部性原理：自然语言的依赖距离分布

自然语言中，token间的依赖关系随距离衰减：
- **近距离依赖**（1-50 tokens）：语法结构、词组搭配，占绝大多数
- **中距离依赖**（50-500 tokens）：句子间指代、段落内逻辑
- **远距离依赖**（500+ tokens）：跨段指代、全局主题，相对较少

研究表明，LLM的注意力权重高度集中在局部窗口内，远距离token的注意力权重通常很小。滑动窗口丢弃这些低权重的远距离注意力，对性能影响有限。

#### 3. 信息传递：多层堆叠扩大感受野

虽然单层只能看到$W$个邻居，但通过多层堆叠，信息可以逐层传递：

```
Layer 1: token_i 可以看到 [i-W+1, i]  → 感受野 = W
Layer 2: token_i 通过Layer 1的输出间接看到 [i-2W+2, i] → 感受野 = 2W
Layer L: 感受野 = L × W
```

例如Mistral-7B：$W=4096$，$L=32$层，理论感受野 = $4096 \times 32 = 131072$ tokens，远超实际序列长度。

### 滑动窗口注意力的数学公式

注意力计算仅对窗口内的KV进行：
$$\text{Attn}(q_i, K, V) = \text{softmax}\left(\frac{q_i K_{[i-W:i]}^T}{\sqrt{d_k}}\right) V_{[i-W:i]}$$

其中：
- $q_i$：第$i$个token的查询向量
- $K_{[i-W:i]}$：第$i-W$到第$i$个token的Key矩阵
- $V_{[i-W:i]}$：对应的Value矩阵
- $W$：窗口大小
- 多层后的有效感受野：$W \times n_{layers}$

### 滑动窗口在Prefill和Decode阶段的不同实现

#### Prefill阶段

Prefill阶段一次处理多个token，需要构造**因果+窗口mask**：
- 因果约束：token $i$ 不能看到 token $j > i$
- 窗口约束：token $i$ 不能看到 token $j < i - W + 1$

合并后的mask为：$M_{ij} = 1$ 当且仅当 $i - W + 1 \leq j \leq i$

```
W=4, N=6 的滑动窗口mask:
     0 1 2 3 4 5
  0 [1 0 0 0 0 0]
  1 [1 1 0 0 0 0]
  2 [1 1 1 0 0 0]
  3 [1 1 1 1 0 0]
  4 [0 1 1 1 1 0]
  5 [0 0 1 1 1 1]
```

#### Decode阶段

Decode阶段每次只有1个query token，天然满足因果约束。KV Cache只需要保留最近$W$个token的KV，超出窗口的自动淘汰：

```
KV Cache: [k_0, k_1, ..., k_{W-1}]  (最多W个)

新token到来时:
  1. 计算新的 k_new, v_new
  2. 拼接到KV Cache: [..., k_{W-1}, k_new]
  3. 如果长度 > W, 裁剪: [k_1, ..., k_{W-1}, k_new]
```

In [ ]:
class SlidingWindowAttention(nn.Module):
    """滑动窗口注意力：产业级实现

    实现要点：
    - Prefill阶段：使用因果mask + 窗口mask，只计算窗口内的注意力
    - Decode阶段：KV Cache只保留最近W个token，自动淘汰旧KV
    - 支持GQA（n_kv_heads可不同于n_heads）
    """
    def __init__(self, dim: int, n_heads: int, n_kv_heads: int, window_size: int):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.window_size = window_size

        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * self.head_dim, dim, bias=False)

    def _make_sliding_window_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """创建因果 + 滑动窗口mask"""
        mask = torch.zeros(seq_len, seq_len, device=device, dtype=torch.bool)
        for i in range(seq_len):
            start = max(0, i - self.window_size + 1)
            mask[i, start:i+1] = True
        return mask

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        if self.n_rep == 1:
            return x
        B, n_kv, S, D = x.shape
        return x[:, :, None, :, :].expand(B, n_kv, self.n_rep, S, D).reshape(B, n_kv * self.n_rep, S, D)

    def forward(
        self,
        x: torch.Tensor,
        cached_k: Optional[torch.Tensor] = None,
        cached_v: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k_new = self.k_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v_new = self.v_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if cached_k is not None and cached_v is not None:
            k_full = torch.cat([cached_k, k_new], dim=2)
            v_full = torch.cat([cached_v, v_new], dim=2)
        else:
            k_full = k_new
            v_full = v_new

        total_len = k_full.shape[2]
        if total_len > self.window_size:
            k_full = k_full[:, :, total_len - self.window_size:, :]
            v_full = v_full[:, :, total_len - self.window_size:, :]
            total_len = self.window_size

        k_expanded = self._repeat_kv(k_full)
        v_expanded = self._repeat_kv(v_full)

        if cached_k is None and N > 1:
            window_mask = self._make_sliding_window_mask(N, x.device)
            attn_output = F.scaled_dot_product_attention(
                q, k_expanded, v_expanded, attn_mask=window_mask
            )
        else:
            attn_output = F.scaled_dot_product_attention(q, k_expanded, v_expanded)

        out = attn_output.transpose(1, 2).reshape(B, N, -1)
        return self.out_proj(out), k_full, v_full

dim, n_heads, n_kv_heads = 256, 8, 2
window_size = 64
swa = SlidingWindowAttention(dim, n_heads, n_kv_heads, window_size)

print(f"=== 滑动窗口注意力 ===")
print(f"dim={dim}, n_heads={n_heads}, n_kv_heads={n_kv_heads}, window_size={window_size}")

x_prefill = torch.randn(1, 32, dim)
with torch.no_grad():
    out, k_cached, v_cached = swa(x_prefill)
print(f"\nPrefill(32 tokens): 输出={out.shape}, KV Cache={k_cached.shape}")

for step in range(40):
    x_decode = torch.randn(1, 1, dim)
    with torch.no_grad():
        out, k_cached, v_cached = swa(x_decode, cached_k=k_cached, cached_v=v_cached)

print(f"Decode(40步后): KV Cache={k_cached.shape}")
print(f"KV Cache长度={k_cached.shape[2]}, 窗口大小={window_size}")
print(f"KV Cache被自动裁剪到窗口大小，不随序列增长")

print(f"\n=== 滑动窗口 vs 全注意力 内存对比 ===")
n_layers = 32
for ws in [256, 512, 1024, 4096]:
    sw_kv_per_layer = 1 * n_kv_heads * ws * (dim // n_heads) * 2 * 2
    sw_total = n_layers * sw_kv_per_layer / 1024 / 1024
    print(f"  窗口W={ws:<5} 总KV Cache={sw_total:.1f}MB (固定，不随序列增长)")

print(f"\n=== 感受野分析 ===")
for ws in [512, 1024, 4096]:
    rf = ws * n_layers
    print(f"  W={ws}, L={n_layers}层 → 有效感受野={rf} tokens")

---
## 2.2.4 稀疏注意力

### 什么是稀疏注意力？

通过选择性地计算部分注意力对，跳过大部分注意力计算，将复杂度从$O(n^2)$降低。稀疏注意力基于一个关键观察：注意力矩阵通常是稀疏的，大部分注意力权重集中在少数位置。

### 为什么稀疏注意力有效？

#### 注意力分布的稀疏性

研究表明LLM的注意力分布高度稀疏。具体来说：

1. **注意力集中现象**：在每一行注意力权重中，通常只有少数几个token获得显著权重（>0.01），其余token的权重接近0。这意味着大部分$QK^T$的计算结果是无效的。

2. **局部性**：自然语言中相邻token的依赖关系最强，局部窗口内的注意力最重要。远距离token的注意力权重通常很小，但并非完全无用。

3. **全局信息**：少数全局token（如[CLS]、句首token、段落首token）可以传递全局信息，弥补局部注意力的不足。

#### 稀疏注意力的核心思想：只计算重要的注意力对

如果注意力矩阵中只有$sp$比例的元素是重要的，那么只计算这些元素：
- 计算复杂度从$O(n^2 d)$降为$O(sp \cdot n^2 d)$
- 内存从$O(n^2)$降为$O(sp \cdot n^2)$
- 当$sp \ll 1$时（如$sp = \sqrt{n}/n$），复杂度可降至$O(n\sqrt{n})$或更低

### 稀疏注意力的常见模式

#### 1. 局部+全局模式（Longformer）

每个token关注：
- **局部窗口**：相邻$w$个token（$O(n \cdot w)$计算量）
- **全局token**：少数特殊token（如[CLS]）关注所有位置，也被所有位置关注（$O(n \cdot g)$计算量）

```
全局token [G] 可以看到所有位置，也被所有位置看到
普通token 只能看到窗口内的邻居 + 全局token

     G t1 t2 t3 t4 t5 t6 t7
  G [1  1  1  1  1  1  1  1]  ← 全局token看到所有
 t1 [1  1  1  0  0  0  0  0]  ← 局部窗口 + 全局
 t2 [1  1  1  1  0  0  0  0]
 t3 [1  0  1  1  1  0  0  0]
 t4 [1  0  0  1  1  1  0  0]
```

#### 2. 膨胀注意力模式（BigBird）

在局部窗口的基础上，以固定间隔采样远距离token：
- **局部窗口**：相邻$w$个token
- **膨胀窗口**：每隔$d$个token采样一个（如token 0, d, 2d, ...）
- **随机窗口**：随机采样少量远距离token

膨胀注意力的优势：通过间隔采样，用更少的计算覆盖更远的位置。

#### 3. 块稀疏模式（Sparse Transformer）

将注意力矩阵分成$\sqrt{n} \times \sqrt{n}$的块，只计算部分块：
- **局部块**：对角线附近的块（局部注意力）
- **全局块**：固定位置的块（如每$\sqrt{n}$个位置取一块）

计算复杂度：$O(n \cdot \sqrt{n})$

#### 4. 滑动窗口（Mistral）

最简单的稀疏模式——固定窗口，已在2.2.3节详细介绍。

### 稀疏注意力的实现挑战

稀疏注意力的理论计算量减少很可观，但**实际加速需要专门的kernel**：

1. **标准GPU kernel不友好**：GPU的矩阵乘法是密集的，稀疏操作会导致大量无效计算和内存访问。

2. **需要Block Sparse kernel**：将稀疏模式转换为块稀疏（block-sparse）操作，利用GPU的块级并行性。Longformer和BigBird都使用了自定义CUDA kernel。

3. **端侧实现更困难**：NPU/CPU的稀疏计算支持更有限，滑动窗口是最实用的稀疏方案。

| 模式 | 计算复杂度 | 代表模型 | 特点 |
|------|-----------|----------|------|
| 局部+全局 | $O(n \cdot w + n \cdot g)$ | Longformer | 局部窗口+全局token |
| 膨胀注意力 | $O(n \cdot n/d)$ | BigBird | 间隔采样，覆盖更远位置 |
| 块稀疏 | $O(n \cdot \sqrt{n})$ | Sparse Transformer | 固定稀疏模式 |
| 滑动窗口 | $O(n \cdot w)$ | Mistral | 固定窗口，最简单高效 |

其中$w$为窗口大小，$g$为全局token数，$d$为膨胀率。

In [ ]:
class LocalGlobalSparseAttention(nn.Module):
    """局部+全局稀疏注意力：Longformer风格

    注意：本实现为教学演示，仍计算完整注意力矩阵后应用mask，
    不减少实际FLOPs。实际稀疏注意力应跳过mask为0的注意力对计算，
    需要自定义CUDA kernel或使用Block Sparse Attention等库。
    """
    def __init__(self, dim, n_heads, window_size=64, n_global_tokens=1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.window_size = window_size
        self.n_global_tokens = n_global_tokens
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def _create_sparse_mask(self, seq_len):
        mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
        for i in range(seq_len):
            start = max(0, i - self.window_size // 2)
            end = min(seq_len, i + self.window_size // 2 + 1)
            mask[i, start:end] = True
            mask[i, :self.n_global_tokens] = True
            mask[:self.n_global_tokens, i] = True
        return mask

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        sparse_mask = self._create_sparse_mask(N).unsqueeze(0).unsqueeze(0)
        attn = attn.masked_fill(~sparse_mask.to(attn.device), float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = attn.masked_fill(attn.isnan(), 0)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

dim, n_heads = 256, 4
sparse_attn = LocalGlobalSparseAttention(dim, n_heads, window_size=64, n_global_tokens=1)
x = torch.randn(1, 128, dim)
with torch.no_grad():
    out = sparse_attn(x)
print(f"=== 局部+全局稀疏注意力 ===")
print(f"输出形状: {out.shape}")

print(f"\n=== 稀疏注意力计算量对比 ===")
print(f"\n{'方法':<30} {'注意力对数':<15} {'占比':<10}")
print("-" * 55)

window_sizes = [32, 64, 128]
for seq_len in [256, 512, 1024]:
    full_pairs = seq_len * seq_len
    print(f"\n序列长度 = {seq_len}")
    print(f"  {'全注意力':<28} {full_pairs:<15,} {'100%':<10}")
    for ws in window_sizes:
        local_pairs = seq_len * ws
        global_pairs = seq_len * 1 + 1 * seq_len
        total_pairs = local_pairs + global_pairs
        ratio = total_pairs / full_pairs * 100
        print(f"  {'局部(ws='+str(ws)+')+全局':<28} {total_pairs:<15,} {ratio:<10.1f}%")

print(f"\n注意: 上述为理论计算量减少，实际加速需要稀疏注意力kernel支持")
print(f"产业界: Longformer/BigBird使用自定义CUDA kernel实现真正的稀疏计算")
print(f"端侧: 滑动窗口注意力是最实用的稀疏方案，实现简单且硬件友好")

---
## 2.2.5 推理中的注意力机制

### Prefill vs Decode：注意力计算的本质差异

LLM推理分为两个阶段，注意力计算的形态完全不同。理解这种差异是选择正确优化策略的基础。

| | Prefill | Decode |
|---|---------|--------|
| **输入** | 完整prompt（多个token） | 单个新token |
| **Q来源** | 所有输入token | 仅最新token |
| **K/V来源** | 所有输入token | 历史KV Cache + 新token |
| **注意力矩阵** | $N \times N$（方形） | $1 \times N$（行向量） |
| **计算瓶颈** | 计算密集（$O(N^2)$） | 内存带宽密集（读KV Cache） |
| **优化重点** | Flash Attention | GQA/KV压缩 |

#### Prefill阶段：计算密集

Prefill阶段一次处理整个prompt（如1024个token），需要计算$1024 \times 1024$的注意力矩阵。这是典型的**计算密集**（compute-bound）操作：
- FLOPs：$O(N^2 \cdot d)$，随序列长度平方增长
- 内存：需要存储$N \times N$的注意力矩阵
- 优化重点：**减少计算量和中间矩阵存储** → Flash Attention

#### Decode阶段：内存带宽密集

Decode阶段每次只处理1个新token，注意力矩阵是$1 \times N$的行向量。计算量很小（$O(N \cdot d)$），但需要**读取完整KV Cache**：
- FLOPs：$O(N \cdot d)$，随序列长度线性增长
- 内存带宽：每步需读取$2 \times n_{kv\_heads} \times N \times d_k$个元素
- 优化重点：**减少KV Cache大小和读取量** → GQA/KV量化

#### 为什么Decode是内存带宽密集的？

一个简化的算术分析：

假设$N=4096$，$d_k=128$，$n_{kv\_heads}=32$，FP16：
- KV Cache大小：$2 \times 32 \times 4096 \times 128 \times 2 = 64$ MB
- 计算量：$2 \times 32 \times 4096 \times 128 = 33.5$ MFLOPs
- GPU A100算力：312 TFLOPS → 计算耗时：$33.5 / 312000 \approx 0.1 \mu s$
- GPU A100带宽：2 TB/s → 读取耗时：$64 / 2000 \approx 32 \mu s$

读取时间是计算时间的**300倍**！GPU的计算单元在等数据，这就是内存带宽瓶颈。

### 因果mask（Causal Mask）

自回归模型中，token只能关注自身及之前的token，不能看到未来信息。因果mask是一个下三角矩阵：
$$M_{ij} = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

#### Prefill阶段的因果mask

Prefill阶段有$N$个query token，需要显式构造$N \times N$的因果mask：

```
N=4 的因果mask:
     0 1 2 3
  0 [0 -∞ -∞ -∞]   token 0 只能看到自己
  1 [0  0 -∞ -∞]   token 1 可以看到 0, 1
  2 [0  0  0 -∞]   token 2 可以看到 0, 1, 2
  3 [0  0  0  0]   token 3 可以看到所有
```

SDPA通过`is_causal=True`参数自动处理因果mask，不需要手动构造。

#### Decode阶段无需因果mask

Decode阶段只有1个query token（最新生成的token），它需要关注所有历史token，天然满足因果约束——没有"未来token"需要mask。因此`is_causal=False`。

In [ ]:
class CausalAttentionWithKVCache(nn.Module):
    """带KV Cache的因果注意力：完整推理流程

    演示Prefill和Decode两个阶段注意力计算的差异：
    - Prefill: 多个token并行计算，需要因果mask
    - Decode: 单token增量计算，KV Cache拼接，无需mask
    """
    def __init__(self, dim: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * self.head_dim, dim, bias=False)

    def _repeat_kv(self, x):
        if self.n_rep == 1:
            return x
        B, n_kv, S, D = x.shape
        return x[:, :, None, :, :].expand(B, n_kv, self.n_rep, S, D).reshape(B, n_kv * self.n_rep, S, D)

    def forward(
        self,
        x: torch.Tensor,
        cached_k: Optional[torch.Tensor] = None,
        cached_v: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k_new = self.k_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v_new = self.v_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if cached_k is not None and cached_v is not None:
            k_full = torch.cat([cached_k, k_new], dim=2)
            v_full = torch.cat([cached_v, v_new], dim=2)
        else:
            k_full = k_new
            v_full = v_new

        k_expanded = self._repeat_kv(k_full)
        v_expanded = self._repeat_kv(v_full)

        is_prefill = (cached_k is None) and (N > 1)
        if is_prefill:
            attn_output = F.scaled_dot_product_attention(
                q, k_expanded, v_expanded, is_causal=True
            )
        else:
            attn_output = F.scaled_dot_product_attention(
                q, k_expanded, v_expanded, is_causal=False
            )

        out = attn_output.transpose(1, 2).reshape(B, N, -1)
        return self.out_proj(out), k_full, v_full

dim, n_heads, n_kv_heads = 256, 8, 2
attn = CausalAttentionWithKVCache(dim, n_heads, n_kv_heads)

print(f"=== Prefill vs Decode 注意力计算对比 ===")
print(f"dim={dim}, n_heads={n_heads}, n_kv_heads={n_kv_heads}")

prompt_len = 16
x_prefill = torch.randn(1, prompt_len, dim)
with torch.no_grad():
    out_prefill, k_cache, v_cache = attn(x_prefill)

print(f"\n--- Prefill阶段 ---")
print(f"输入: {prompt_len} tokens")
print(f"Q shape: (1, {n_heads}, {prompt_len}, {dim//n_heads})")
print(f"K/V shape: (1, {n_kv_heads}, {prompt_len}, {dim//n_heads})")
print(f"注意力矩阵: ({prompt_len}, {prompt_len}) → 需要{prompt_len*prompt_len}次计算")
print(f"需要因果mask: 是 (下三角)")
print(f"KV Cache: K={k_cache.shape}, V={v_cache.shape}")

print(f"\n--- Decode阶段 (逐步生成) ---")
decode_steps = 8
for step in range(decode_steps):
    x_decode = torch.randn(1, 1, dim)
    with torch.no_grad():
        out_decode, k_cache, v_cache = attn(x_decode, cached_k=k_cache, cached_v=v_cache)
    total_kv_len = k_cache.shape[2]
    print(f"  Step {step+1}: Q=(1,{n_heads},1,{dim//n_heads}), "
          f"K/V=(1,{n_kv_heads},{total_kv_len},{dim//n_heads}), "
          f"注意力=(1,{total_kv_len})")

print(f"\n--- 关键差异总结 ---")
print(f"Prefill: 注意力矩阵是方阵(N×N), 计算密集, 需要因果mask")
print(f"Decode: 注意力矩阵是行向量(1×N), 内存带宽密集, 无需mask")
print(f"Decode的瓶颈: 每步都要读取完整KV Cache, KV越大越慢")
print(f"→ 这就是GQA/KV压缩对Decode阶段如此重要的原因")

---
## 2.2.6 产业级实战：完整推理流程

将GQA、SDPA、KV Cache、因果mask组合成一个完整的Transformer推理流程，模拟真实LLM推理引擎的核心逻辑。

### 完整推理流程的架构

一个产业级LLM推理引擎的注意力计算流程：

```
输入: input_ids (prompt或新生成的token)
  ↓
Token Embedding
  ↓
对每一层:
  ├─ LayerNorm (Pre-Norm)
  ├─ GQA注意力:
  │   ├─ Q/K/V投影 (Q: n_heads组, K/V: n_kv_heads组)
  │   ├─ KV Cache拼接 (Decode时与历史KV拼接)
  │   ├─ KV头扩展 (expand, 不复制内存)
  │   ├─ SDPA计算 (Prefill: is_causal=True, Decode: is_causal=False)
  │   └─ 输出投影
  ├─ 残差连接
  ├─ LayerNorm
  ├─ FFN (SiLU激活)
  └─ 残差连接
  ↓
Final LayerNorm
  ↓
LM Head → logits → 采样下一个token
```

### 产业级推理引擎的关键优化点

1. **PagedAttention**（vLLM）：将KV Cache分页管理，避免内存碎片，支持更大的batch size
2. **Continuous Batching**：不等所有序列完成再处理新请求，而是动态调度
3. **KV Cache量化**：将FP16的KV Cache量化为INT8/INT4，减少内存带宽
4. **Speculative Decoding**：用小模型预测多个token，大模型并行验证，加速Decode

In [ ]:
class TransformerBlock(nn.Module):
    """Transformer Block：GQA + FFN + 残差连接"""
    def __init__(self, dim: int, n_heads: int, n_kv_heads: int, ffn_dim_multiplier: float = 4.0):
        super().__init__()
        self.attn_norm = nn.LayerNorm(dim)
        self.attn = CausalAttentionWithKVCache(dim, n_heads, n_kv_heads)
        self.ffn_norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, int(dim * ffn_dim_multiplier), bias=False),
            nn.SiLU(),
            nn.Linear(int(dim * ffn_dim_multiplier), dim, bias=False),
        )

    def forward(self, x, cached_k=None, cached_v=None):
        h = self.attn_norm(x)
        attn_out, k_updated, v_updated = self.attn(h, cached_k, cached_v)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, k_updated, v_updated

class SimpleTransformerLM(nn.Module):
    """简化Transformer语言模型：模拟LLaMA/Mistral的核心结构

    产业级特性：
    - GQA (Grouped-Query Attention)
    - SDPA (scaled_dot_product_attention)
    - KV Cache增量更新
    - Pre-norm (LayerNorm在注意力/FFN之前)
    - SiLU激活函数
    """
    def __init__(
        self,
        vocab_size: int = 32000,
        dim: int = 512,
        n_layers: int = 4,
        n_heads: int = 8,
        n_kv_heads: int = 2,
        max_seq_len: int = 2048,
    ):
        super().__init__()
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads

        self.token_emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            TransformerBlock(dim, n_heads, n_kv_heads) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(
        self,
        input_ids: torch.Tensor,
        kv_caches: Optional[list] = None,
    ) -> Tuple[torch.Tensor, list]:
        B, N = input_ids.shape
        h = self.token_emb(input_ids)

        new_kv_caches = []
        for i, layer in enumerate(self.layers):
            cached_k = kv_caches[i][0] if kv_caches is not None else None
            cached_v = kv_caches[i][1] if kv_caches is not None else None
            h, k_updated, v_updated = layer(h, cached_k, cached_v)
            new_kv_caches.append((k_updated, v_updated))

        logits = self.lm_head(self.final_norm(h))
        return logits, new_kv_caches

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 20,
        temperature: float = 1.0,
        top_k: int = 50,
    ) -> torch.Tensor:
        kv_caches = None
        generated = input_ids

        logits, kv_caches = self.forward(input_ids, kv_caches=None)
        next_token = self._sample_next(logits[:, -1, :], temperature, top_k)
        generated = torch.cat([generated, next_token], dim=-1)

        for _ in range(max_new_tokens - 1):
            logits, kv_caches = self.forward(next_token, kv_caches=kv_caches)
            next_token = self._sample_next(logits[:, -1, :], temperature, top_k)
            generated = torch.cat([generated, next_token], dim=-1)

        return generated

    def _sample_next(self, logits, temperature, top_k):
        logits = logits / max(temperature, 1e-8)
        if top_k > 0:
            top_k_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < top_k_values[:, -1:]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, num_samples=1)

    def kv_cache_memory_mb(self, seq_len: int) -> float:
        kv_per_layer = 1 * self.n_kv_heads * seq_len * self.head_dim * 2 * 2
        return self.n_layers * kv_per_layer / 1024 / 1024

vocab_size = 1000
model = SimpleTransformerLM(
    vocab_size=vocab_size, dim=256, n_layers=4, n_heads=8, n_kv_heads=2
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"=== 产业级Transformer推理模型 ===")
print(f"参数量: {total_params:,}")
print(f"结构: {model.n_layers}层, {model.n_heads}Q头, {model.n_kv_heads}KV头 (GQA)")

prompt = torch.randint(0, vocab_size, (1, 16))
generated = model.generate(prompt, max_new_tokens=10, temperature=0.8, top_k=50)
print(f"\nPrompt长度: {prompt.shape[1]}")
print(f"生成长度: {generated.shape[1]}")
print(f"生成token IDs: {generated[0, prompt.shape[1]:].tolist()}")

print(f"\n=== KV Cache内存分析 ===")
for seq_len in [128, 256, 512, 1024, 2048]:
    mem = model.kv_cache_memory_mb(seq_len)
    mha_mem = model.n_layers * 1 * model.n_heads * seq_len * model.head_dim * 2 * 2 / 1024 / 1024
    print(f"  seq={seq_len:<5} GQA={mem:.2f}MB MHA={mha_mem:.2f}MB 节省{(1-mem/mha_mem)*100:.0f}%")

In [ ]:
print(f"=== Prefill vs Decode 性能剖析 ===")

model = SimpleTransformerLM(
    vocab_size=1000, dim=256, n_layers=4, n_heads=8, n_kv_heads=2
)
model.eval()

prompt_len = 64
decode_steps = 20
prompt = torch.randint(0, vocab_size, (1, prompt_len))

with torch.no_grad():
    start = time.perf_counter()
    logits, kv_caches = model(prompt, kv_caches=None)
    prefill_time = time.perf_counter() - start

    decode_times = []
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    for _ in range(decode_steps):
        start = time.perf_counter()
        logits, kv_caches = model(next_token, kv_caches=kv_caches)
        decode_times.append(time.perf_counter() - start)
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

avg_decode_time = np.mean(decode_times)
total_decode_time = sum(decode_times)
total_time = prefill_time + total_decode_time

print(f"Prompt长度: {prompt_len} tokens")
print(f"Decode步数: {decode_steps}")
print(f"\n{'阶段':<15} {'耗时(ms)':<15} {'占总时间':<15} {'每token耗时':<15}")
print("-" * 60)
print(f"{'Prefill':<15} {prefill_time*1000:<15.2f} {prefill_time/total_time*100:<15.1f}% {prefill_time/prompt_len*1000:<15.3f}ms")
print(f"{'Decode(总)':<15} {total_decode_time*1000:<15.2f} {total_decode_time/total_time*100:<15.1f}% {avg_decode_time*1000:<15.3f}ms")

print(f"\n=== 真实LLM推理瓶颈分析 ===")
print(f"短prompt+长生成: Decode阶段占主导 → 优化KV Cache读取(GQA/量化)")
print(f"长prompt+短生成: Prefill阶段占主导 → 优化注意力计算(Flash Attention)")
print(f"\n产业界优化组合:")
print(f"  Prefill优化: Flash Attention (SDPA自动启用)")
print(f"  Decode优化: GQA减少KV Cache + KV量化减少带宽")
print(f"  两者结合: SDPA + GQA + KV Cache量化 = 端侧推理最优解")

---
## 2.2.7 注意力优化方法综合对比

不同注意力优化方法在计算复杂度、精度影响和硬件需求方面的对比。实际部署中通常组合使用多种方法。

### 优化方法的组合策略

不同的优化方法作用于推理的不同阶段和不同层面，可以组合使用：

```
推理阶段    计算层面      优化方法              效果
─────────────────────────────────────────────────────────
Prefill    注意力计算    Flash Attention       减少HBM访问，加速2-4x
Prefill    注意力架构    GQA                   减少KV投影参数
Prefill    注意力模式    滑动窗口/稀疏         减少计算量
Decode     KV Cache     GQA                   减少KV Cache大小
Decode     KV Cache     KV量化                减少读取带宽
Decode     KV Cache     滑动窗口              固定Cache大小
Decode     注意力计算    SDPA                  自动选择最优后端
```

### 产业级优化组合推荐

| 场景 | 推荐组合 | 原因 |
|------|---------|------|
| GPU端侧 | SDPA + GQA | 精度与效率最优平衡 |
| CPU/NPU端侧 | GQA + 滑动窗口 | 减少KV Cache和计算量 |
| 长序列 | 滑动窗口注意力 | 固定KV Cache，线性复杂度 |
| 极致速度 | MQA | 最小KV Cache，适合短序列 |
| 通用推荐 | SDPA + GQA + KV量化 | 覆盖Prefill和Decode两个阶段 |

In [ ]:
print(f"{'方法':<25} {'复杂度':<15} {'精度影响':<15} {'硬件需求':<15} {'适用阶段':<15}")
print("-" * 85)
methods = [
    ("Flash Attention", "O(N²)计算", "无(精确)", "GPU SRAM", "Prefill"),
    ("MQA", "O(N²)计算", "较小", "通用", "Decode"),
    ("GQA", "O(N²)计算", "很小", "通用", "Decode"),
    ("滑动窗口", "O(N·W)", "丢失远距", "通用", "Prefill+Decode"),
    ("局部+全局", "O(N·W+N)", "较小", "通用", "Prefill+Decode"),
    ("稀疏模式", "O(N·√N)", "中等", "通用", "Prefill"),
]
for name, comp, impact, hw, stage in methods:
    print(f"{name:<25} {comp:<15} {impact:<15} {hw:<15} {stage:<15}")

print(f"\n=== 真实模型配置参考 ===")
models = [
    ("LLaMA-2-70B", "GQA(8/64)", "SDPA", "无"),
    ("LLaMA-3-8B", "GQA(8/32)", "SDPA", "无"),
    ("LLaMA-3-70B", "GQA(8/64)", "SDPA", "无"),
    ("Mistral-7B", "GQA(8/32)+SW", "SDPA", "滑动窗口W=4096"),
    ("Mixtral-8x7B", "GQA(8/32)+SW", "SDPA", "滑动窗口W=4096"),
    ("Phi-3-mini", "GQA(8/32)", "SDPA", "无"),
]
print(f"\n{'模型':<20} {'注意力类型':<20} {'计算后端':<10} {'额外优化':<15}")
print("-" * 65)
for name, attn_type, backend, extra in models:
    print(f"{name:<20} {attn_type:<20} {backend:<10} {extra:<15}")